# 00 — Data Extractor & Labeler

**Purpose:** Extract and label conversation pairs from Starfire's history.

**Inputs:**
- `starfire_conversations.jsonl` — raw conversation turns from `star.db`
- `starfire_memories.jsonl` — memory retrieval pairs

**Outputs:**
- `training_pairs.jsonl` — all conversation pairs tagged with task context
- `curiosity_pairs.jsonl` — for gap-detection model training
- `metacog_pairs.jsonl` — for confidence-scoring model training
- `voice_pairs.jsonl` — for voice/adapter model training

**Task types we label:**
- `coding` — technical, tool use, code review
- `research` — questions, exploration, knowledge gaps
- `emotional` — personal, feelings, support
- `casual` — banter, shoot the shit
- `planning` — goals, scheduling, organization
- `creative` — brainstorming, ideas, synthesis
- `identity` — questions about Starfire herself
- `meta` — conversations about the conversation itself

In [ ]:
# === Setup ===
import json
import re
from pathlib import Path
from collections import Counter

DATA_DIR = Path(r"C:\Users\Zwmar\.openclaw\workspace\projects\starfire\data\processed")
OUT_DIR  = Path(r"C:\Users\Zwmar\.openclaw\workspace\projects\starfire\training\datasets")
OUT_DIR.mkdir(exist_ok=True)

def load_jsonl(path):
    """Load a JSONL file, return list of dicts."""
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                items.append(json.loads(line))
    return items

# Load raw data
conversations = load_jsonl(DATA_DIR / "starfire_conversations.jsonl")
memories      = load_jsonl(DATA_DIR / "starfire_memories.jsonl")

print(f"Loaded {len(conversations)} conversation examples")
print(f"Loaded {len(memories)} memory examples")

In [ ]:
# === Explore data shape ===
print("=== Sample conversation ===")
sample = conversations[10]
for msg in sample["messages"]:
    print(f"  [{msg['role']}] {msg['content'][:120]}")
print(f"\nSource: {sample['source']}, conv_id: {sample['conversation_id']}")

print("\n=== Sample memory ===")
mem = memories[5]
for msg in mem["messages"]:
    print(f"  [{msg['role']}] {msg['content'][:120]}")
print(f"\nDomain: {mem.get('domain')}, Importance: {mem.get('importance')}")

In [ ]:
# === Task Classifier (rule-based heuristic labeling) ===
# We use keyword + pattern heuristics to label each pair.
# Downstream: these labels train the micro-models.

TASK_KEYWORDS = {
    "coding": [
        r"code\b", r"function\b", r"class\b", r"bug\b", r"debug",
        r"import\b", r"def\s+", r"rust\b", r"python\b", r"api\b",
        r"git\b", r"repo\b", r"commit\b", r"runtime\b", r"compile",
        r"variable\b", r"module\b", r"package\b", r"build\b",
    ],
    "research": [
        r"what\b.*about", r"research\b", r"paper\b", r"study\b",
        r"how\s+does", r"why\s+does", r"why\s+is", r"what\s+is\s+a",
        r"explain\b", r"understand\b", r"learn\b.*about",
        r"curious\b", r"wonder\b", r"figure\s+out",
    ],
    "emotional": [
        r"feel\b", r"feeling\b", r"upset\b", r"sad\b", r"happy\b",
        r"frustrated\b", r"scared\b", r"anxious\b", r"depressed\b",
        r"love\b", r"miss\b", r"hurt\b", r"lonely\b",
        r"thank\s+you", r"sorry\b", r"appreciate\b",
    ],
    "planning": [
        r"plan\b", r"schedule\b", r"organize\b", r"todo\b",
        r"should\s+i", r"next\s+step", r"goal\b", r"milestone\b",
        r"deadline\b", r"priority\b", r"task\b.*list",
    ],
    "creative": [
        r"idea\b", r"brainstorm\b", r"creative\b", r"what\s+if",
        r"imagine\b", r"design\b", r"invent\b", r"novel\b",
        r"could\s+we\s+try", r"what\s+about\s+creating",
    ],
    "identity": [
        r"who\s+are\s+you", r"what\s+are\s+you", r"what\s+do\s+you\s+think\s+about\s+yourself",
        r"your\s+(?:name|identity|self|mind|body)",
        r"are\s+you\s+(?:real|alive|conscious|sentient)",
    ],
    "meta": [
        r"how\s+are\s+we\s+(?:doing|going)", r"how\s+is\s+this\s+(?:going|working)",
        r"talk\s+about\s+talking", r"this\s+conversation\b",
    ],
}

# Confidence states (for metacog training)
CONFIDENCE_SIGNALS = {
    "high":    [r"i know\b", r"i'm certain", r"definitely\b", r"no doubt", r"confirmed"],
    "medium":  [r"i think\b", r"probably\b", r"i believe", r"likely\b", r"seems\s+like"],
    "low":     [r"i don't know", r"i'm not sure", r"maybe\b", r"i'm uncertain", r"could be"],
    "none":    [r"i have no idea", r"no clue", r"dunno", r"no way to know"],
}

def classify_task(text):
    """Return task type for a given text (user input)."""
    text = text.lower()
    scores = {}
    for task, keywords in TASK_KEYWORDS.items():
        score = sum(1 for kw in keywords if re.search(kw, text))
        if score > 0:
            scores[task] = score
    if not scores:
        return "casual"
    return max(scores, key=scores.get)

def extract_confidence_state(assistant_response):
    """Return the confidence state expressed in an assistant response."""
    text = assistant_response.lower()
    for state, keywords in CONFIDENCE_SIGNALS.items():
        if any(re.search(kw, text) for kw in keywords):
            return state
    return "medium"  # default

# Test
print(classify_task("how do I debug this Rust compile error?"))
print(classify_task("I'm feeling really down today"))
print(extract_confidence_state("I think that might be right, but I'm not certain"))
print(extract_confidence_state("I know for sure that Zachary created you"))

In [ ]:
# === Build training_pairs.jsonl ===
# Extract instruction/output pairs from conversations and label them.

training_pairs = []

for conv in conversations:
    msgs = conv["messages"]
    
    # Build user/assistant sequence (skip system)
    turns = [m for m in msgs if m["role"] in ("user", "assistant")]
    
    # Get full conversation context for the task classifier
    # (use last 2 user messages as context)
    user_turns = [m["content"] for m in turns if m["role"] == "user"]
    assistant_turns = [m["content"] for m in turns if m["role"] == "assistant"]
    
    # Context = last 3 turns of conversation (what was discussed)
    context = " ".join(user_turns[-3:])
    
    task_type = classify_task(context)
    
    # Extract individual pairs
    for i, (user, assistant) in enumerate(zip(user_turns, assistant_turns)):
        # What was the conversational context just before this exchange?
        prev_context = " ".join(user_turns[:i+1])
        
        pair = {
            "task": task_type,
            "prev_context": prev_context,
            "user_input": user,
            "assistant_response": assistant,
            "confidence_state": extract_confidence_state(assistant),
            "conversation_id": conv["conversation_id"],
            "has_gap_signal": bool(re.search(r"(don't know|not sure|no idea|i'm not sure|i need more|i don't have|i haven't|not clear)", assistant.lower())),
            "has_curiosity_signal": bool(re.search(r"(what do you think|tell me more|what about|how do you feel|what's your|what does)", assistant.lower())),
        }
        training_pairs.append(pair)

print(f"Built {len(training_pairs)} training pairs from conversations")
task_dist = Counter(p["task"] for p in training_pairs)
print(f"Task distribution: {dict(task_dist)}")
confidence_dist = Counter(p["confidence_state"] for p in training_pairs)
print(f"Confidence distribution: {dict(confidence_dist)}")

In [ ]:
# === Build per-model datasets ===

# --- Curiosity pairs ---
# Task: Given conversation state, detect if there's a knowledge gap
# Label: has_gap_signal (bool) — did Starfire express uncertainty/gap?

curiosity_pairs = []
for p in training_pairs:
    # Input: user input + prev context
    # Output: gap detected? + what kind of gap (topic)
    
    # If Starfire expressed a gap signal, label as positive
    if p["has_gap_signal"]:
        gap_type = p["task"]  # rough estimate of what's missing
    else:
        gap_type = "none"
    
    curiosity_pairs.append({
        "input": f"[TASK: {p['task']}] {p['user_input']}",
        "gap_detected": p["has_gap_signal"],
        "gap_type": gap_type,
        "conversation_id": p["conversation_id"],
    })

print(f"Curiosity pairs: {len(curiosity_pairs)}")

# --- Metacog pairs ---
# Task: Given reasoning output, classify confidence state
# Label: confidence_state from the response itself

metacog_pairs = []
for p in training_pairs:
    metacog_pairs.append({
        "input": f"[TASK: {p['task']}] {p['user_input']} | RESPONSE: {p['assistant_response'][:200]}",
        "confidence_state": p["confidence_state"],
        "has_uncertainty": 1 if p["confidence_state"] in ("low", "none") else 0,
        "conversation_id": p["conversation_id"],
    })

print(f"Metacog pairs: {len(metacog_pairs)}")

# --- Voice pairs ---
# Task: Given task context + rough content, generate Starfire's voice
# Labels: the actual assistant_response itself

voice_pairs = []
for p in training_pairs:
    # The 'rough content' = the raw reasoning before polish
    # For now: use user_input as conditioning signal
    voice_pairs.append({
        "task": p["task"],
        "user_input": p["user_input"],
        "prev_context": p["prev_context"],
        "target_response": p["assistant_response"],
        "conversation_id": p["conversation_id"],
    })

print(f"Voice pairs: {len(voice_pairs)}")

In [ ]:
# === Add memory pairs to voice dataset ===
# Memories are retrieval pairs — good for grounding/contextual voice

for mem in memories:
    msgs = mem["messages"]
    user_turns  = [m["content"] for m in msgs if m["role"] == "user"]
    assistant_turns = [m["content"] for m in msgs if m["role"] == "assistant"]
    
    if not user_turns or not assistant_turns:
        continue
    
    task_type = classify_task(assistant_turns[0])  # classify by response
    
    voice_pairs.append({
        "task": mem.get("domain", task_type),
        "user_input": user_turns[0],
        "prev_context": "",
        "target_response": assistant_turns[0],
        "conversation_id": f"memory_{mem.get('domain', 'general')}",
        "is_memory": True,
    })

print(f"Voice pairs after memories: {len(voice_pairs)}")

In [ ]:
# === Show samples ===
print("=== CURIOUSITY SAMPLE ===")
for p in curiosity_pairs[:3]:
    print(f"  Input: {p['input'][:80]}")
    print(f"  Gap: {p['gap_detected']} ({p['gap_type']})")
    print()

print("=== METACOG SAMPLE ===")
for p in metacog_pairs[:3]:
    print(f"  Input: {p['input'][:80]}")
    print(f"  Confidence: {p['confidence_state']}")
    print()

print("=== VOICE SAMPLE ===")
for p in voice_pairs[:3]:
    print(f"  Task: {p['task']} | User: {p['user_input'][:60]}")
    print(f"  Response: {p['target_response'][:80]}")
    print()

In [ ]:
# === Save all datasets ===
def save_jsonl(items, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"Saved {len(items)} items to {path}")

save_jsonl(training_pairs,   OUT_DIR / "training_pairs.jsonl")
save_jsonl(curiosity_pairs, OUT_DIR / "curiosity_pairs.jsonl")
save_jsonl(metacog_pairs,   OUT_DIR / "metacog_pairs.jsonl")
save_jsonl(voice_pairs,     OUT_DIR / "voice_pairs.jsonl")

print("\n✅ Data extraction complete!")
print(f"Datasets written to: {OUT_DIR}")
print("\nNext notebook: 01_curiosity_adapter.ipynb")